In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

### Loading the Dataset:

In [2]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Converting Text to Binary Presence Features:

The key change from the Multinomial notebook: passing `binary=True` to `CountVectorizer`. Instead of counting occurrences, every cell becomes exactly 0 or 1 — whether that word appears in the message at all.

In [3]:
X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

# binary=True converts counts into presence/absence indicators
vectorizer = CountVectorizer(binary=True)
X_train_bin = vectorizer.fit_transform(X_train)
X_test_bin = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Training matrix shape:", X_train_bin.shape)

# Confirm every value is 0 or 1, not a raw count
print("Unique values in training matrix:", np.unique(X_train_bin.toarray()))

Vocabulary size: 7741
Training matrix shape: (4457, 7741)
Unique values in training matrix: [0 1]


### Fitting Bernoulli Naive Bayes:

In [4]:
bnb = BernoulliNB()
bnb.fit(X_train_bin, y_train)

y_pred_bnb = bnb.predict(X_test_bin)

print("Accuracy:", accuracy_score(y_test, y_pred_bnb))
print("Precision:", precision_score(y_test, y_pred_bnb, pos_label='spam'))
print("Recall:", recall_score(y_test, y_pred_bnb, pos_label='spam'))
print("F1-Score:", f1_score(y_test, y_pred_bnb, pos_label='spam'))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_bnb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_bnb))

Accuracy: 0.9766816143497757
Precision: 0.992
Recall: 0.8322147651006712
F1-Score: 0.9051094890510949

Confusion Matrix:
 [[965   1]
 [ 25 124]]

Classification Report:
               precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       0.99      0.83      0.91       149

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### Direct Comparison — Bernoulli vs. Multinomial on the Same Dataset:

Fitting Multinomial NB again here (on word-count features, not binary) side by side, so both variants are compared under identical train/test splits.

In [5]:
# Multinomial NB uses raw counts, not binary presence
count_vectorizer = CountVectorizer()
X_train_counts = count_vectorizer.fit_transform(X_train)
X_test_counts = count_vectorizer.transform(X_test)

mnb = MultinomialNB()
mnb.fit(X_train_counts, y_train)
y_pred_mnb = mnb.predict(X_test_counts)

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (spam)', 'Recall (spam)', 'F1-Score (spam)'],
    'Bernoulli NB': [
        accuracy_score(y_test, y_pred_bnb),
        precision_score(y_test, y_pred_bnb, pos_label='spam'),
        recall_score(y_test, y_pred_bnb, pos_label='spam'),
        f1_score(y_test, y_pred_bnb, pos_label='spam')
    ],
    'Multinomial NB': [
        accuracy_score(y_test, y_pred_mnb),
        precision_score(y_test, y_pred_mnb, pos_label='spam'),
        recall_score(y_test, y_pred_mnb, pos_label='spam'),
        f1_score(y_test, y_pred_mnb, pos_label='spam')
    ]
})

comparison

,Metric,Bernoulli NB,Multinomial NB
0,Accuracy,0.976682,0.982960
1,Precision (spam),0.992000,0.964286
2,Recall (spam),0.832215,0.906040
3,F1-Score (spam),0.905109,0.934256


### Why the Difference Matters — A Concrete Example:

Consider two hypothetical spam messages: one that says "free" once, another that says "free free free free free". Multinomial NB treats the second as much stronger evidence of spam (5x the count). Bernoulli NB treats them identically — both simply have "free" present.

The cell below finds real short vs. long messages containing a common spam-indicator word, to illustrate how repeated words are treated differently by each model.

In [6]:
# Find a message where a word repeats multiple times, vs. a message where it appears just once
word = 'free'
word_idx = count_vectorizer.vocabulary_.get(word)

if word_idx is not None:
    counts_for_word = X_train_counts[:, word_idx].toarray().ravel()

    # Message with the highest repeat count of this word
    max_count_idx = counts_for_word.argmax()
    # Any message with exactly one occurrence
    single_occurrence_idx = (counts_for_word == 1).nonzero()[0][0]

    print(f"Word being compared: '{word}'\n")

    print(f"Message with count={counts_for_word[max_count_idx]}:")
    print(f"  \"{X_train.iloc[max_count_idx][:80]}...\"")
    print(f"  Multinomial sees count: {counts_for_word[max_count_idx]}")
    print(f"  Bernoulli sees presence: 1 (same as any other message containing '{word}')\n")

    print(f"Message with count={counts_for_word[single_occurrence_idx]}:")
    print(f"  \"{X_train.iloc[single_occurrence_idx][:80]}...\"")
    print(f"  Multinomial sees count: {counts_for_word[single_occurrence_idx]}")
    print(f"  Bernoulli sees presence: 1 (identical signal to the message above)")


Word being compared: 'free'

Message with count=3:
  "FREE MESSAGE Activate your 500 FREE Text Messages by replying to this message wi..."
  Multinomial sees count: 3
  Bernoulli sees presence: 1 (same as any other message containing 'free')

Message with count=1:
  "500 free text msgs. Just text ok to 80488 and we'll credit your account..."
  Multinomial sees count: 1
  Bernoulli sees presence: 1 (identical signal to the message above)


### Summary:

- Bernoulli NB models word **presence/absence**, not frequency — repetition of a word carries no extra weight, unlike Multinomial NB.
- Bernoulli NB also explicitly factors in the *absence* of words that don't appear, which Multinomial NB does not — this can help or hurt depending on the dataset.
- On this dataset, the two variants perform similarly overall, but the underlying probability model each uses is meaningfully different — the right choice depends on whether word repetition is a genuinely useful signal for the classification task at hand.